# Análisis de Default con Regresión Logística

En este notebook realizaremos un análisis completo de clasificación usando Regresión Logística para predecir si un cliente hará default (impago) en base a sus características financieras.

## 1. Importación de Librerías

Importamos las librerías necesarias para el análisis:
- **pandas**: para manipulación de datos
- **numpy**: para operaciones numéricas
- **sklearn**: para el modelo de regresión logística y métricas de evaluación

In [ ]:
# Librerías para manipulación de datos
import pandas as pd
import numpy as np

# Librería para división de datos
from sklearn.model_selection import train_test_split

# Modelo de clasificación
from sklearn.linear_model import LogisticRegression

# Métricas de evaluación
from sklearn.metrics import accuracy_score, confusion_matrix

## 2. Carga de Datos

Cargamos el dataset Default.csv que contiene información sobre clientes y si han incumplido con sus pagos (default). El archivo está separado por tabulaciones (`\t`).

In [ ]:
# Cargar el dataset con separador de tabulación
data = pd.read_csv('Default.csv', sep='\t')

# Mostrar las primeras 5 filas para entender la estructura       
data.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


## 3. Análisis Exploratorio - Distribución de Default

Verificamos cuántos clientes han hecho default vs. los que no, para entender el balance de clases en nuestro dataset.

In [ ]:
# Contar la frecuencia de cada clase en la variable objetivo
# Esto nos ayuda a identificar si hay desbalance de clases
data.default.value_counts()

default
No     9667
Yes     333
Name: count, dtype: int64

## 4. Codificación de Variables Categóricas - Student

Creamos variables dummy para la columna `student` (si es estudiante o no). Usamos `drop_first=True` para evitar **multicolinealidad** (también conocido como "dummy variable trap").

**¿Por qué drop_first=True?**
- Si tenemos 2 categorías (Yes/No), solo necesitamos 1 variable dummy
- La segunda categoría está implícita: si student_Yes = 0, entonces student_No = 1
- Esto evita redundancia y problemas de inversión de matriz en el modelo

In [ ]:
# Crear variables dummy para la columna student
# drop_first=True: elimina una categoría para evitar multicolinealidad (dummy variable trap)
# prefix='student': añade prefijo para identificar fácilmente la variable
# dtype=int: convierte a enteros (0 y 1) en lugar de booleanos
students_dummy = pd.get_dummies(data.student, drop_first=True, prefix='student', dtype=int)
students_dummy.head(3)

,student_Yes
0,0
1,1
2,0


## 5. Integración de Variables Dummy al Dataset

Concatenamos las variables dummy creadas con el dataset original para tener todas las características en un solo DataFrame.

In [ ]:
# Concatenar las variables dummy al dataframe original
# axis=1: concatenar por columnas (horizontalmente)
data = pd.concat([data, students_dummy], axis=1)

# Verificar que la nueva columna se agregó correctamente
data.head(3)

,default,student,balance,income,student_Yes
0,No,No,729.526495,44361.625074,0
1,No,Yes,817.180407,12106.134700,1
2,No,No,1073.549164,31767.138947,0


## 6. Definición de Variables Predictoras (X) y Variable Objetivo (y)

Separamos nuestras características (features):
- **X**: balance, income, student_Yes (variables independientes)
- **y**: default (variable dependiente que queremos predecir)

In [ ]:
# Definir matriz de características (features) - Variables independientes
X = data[['balance', 'income', 'student_Yes']]

# Definir variable objetivo (target) - Variable dependiente
y = data['default']

## 7. División de Datos en Entrenamiento y Prueba

Dividimos los datos en conjuntos de entrenamiento (75%) y prueba (25%). 

**Parámetros importantes:**
- `stratify=y`: Mantiene la misma proporción de clases en ambos conjuntos. Crucial cuando las clases están desbalanceadas.
- `random_state=12`: Semilla aleatoria para garantizar reproducibilidad de los resultados.

**¿Por qué es importante el stratify?**
En nuestro caso, solo ~3.33% de los clientes hacen default. Sin stratify, podríamos tener conjuntos con proporciones muy diferentes, lo que afectaría el entrenamiento y la evaluación.

In [ ]:
# Dividir datos: 75% entrenamiento, 25% prueba
# stratify=y: mantiene la proporción de clases en ambos conjuntos (importante para datos desbalanceados)
# random_state=12: semilla para reproducibilidad de resultados
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=12)

## 8. Verificación de Proporciones - Dataset Completo

Calculamos la proporción de clientes con default en el dataset completo para usarla como referencia.

In [ ]:
# Calcular la proporción de defaults en el dataset completo
# Esto nos da una línea base: ~3.33% de defaults
data.default.value_counts()['Yes'] / data.shape[0]

np.float64(0.0333)

## 9. Verificación de Proporciones - Conjunto de Entrenamiento

Verificamos que la proporción de default en el conjunto de entrenamiento se mantenga similar al dataset completo.

In [ ]:
# Verificar proporción en conjunto de entrenamiento
# Debe ser similar al dataset completo (~3.33%) gracias a stratify
y_train.value_counts()['Yes'] / y_train.shape[0]

np.float64(0.03333333333333333)

## 10. Verificación de Proporciones - Conjunto de Prueba

Verificamos que la proporción de default en el conjunto de prueba también se mantenga similar, confirmando una división estratificada correcta.

In [ ]:
# Verificar proporción en conjunto de prueba
# También debe ser similar al dataset completo (~3.33%)
y_test.value_counts()['Yes'] / y_test.shape[0]

np.float64(0.0332)

## 11. Entrenamiento del Modelo de Regresión Logística

Creamos y entrenamos el modelo de Regresión Logística sin penalización (`penalty=None`).

**Concepto clave - Regresión Logística:**
- No es un modelo de regresión, sino de **clasificación binaria**
- Predice la probabilidad de pertenecer a una clase usando la función sigmoide
- Transforma una combinación lineal en probabilidades entre 0 y 1

**Función del modelo:**
$$P(default = Yes) = \frac{1}{1 + e^{-(β_0 + β_1 \cdot balance + β_2 \cdot income + β_3 \cdot student)}}$$

**Penalty=None** significa que no aplicamos regularización (L1 o L2), permitiendo que el modelo aprenda libremente de los datos.

In [ ]:
# Crear instancia del modelo de Regresión Logística
# penalty=None: sin regularización (L1 o L2), el modelo aprende directamente de los datos
logistic_regression = LogisticRegression(penalty=None)

# Entrenar el modelo con los datos de entrenamiento
# El modelo aprende los coeficientes óptimos que minimizan la función de pérdida
logistic_regression.fit(X_train, y_train)

/Users/leoalvarez/Documents/Digital House Data Science/Ejercicio 2 Clasificadores lineales/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",None
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

## 12. Predicciones sobre el Conjunto de Prueba

Realizamos predicciones sobre el conjunto de prueba. El modelo predice la clase (Yes/No) para cada observación.

In [ ]:
# Realizar predicciones sobre el conjunto de prueba
# El modelo asigna la clase con mayor probabilidad (Yes o No)
y_test_pred = logistic_regression.predict(X_test)
y_test_pred

array(['No', 'No', 'No', ..., 'No', 'No', 'No'],
      shape=(2500,), dtype=object)

## 13. Probabilidades de Predicción

Obtenemos las probabilidades de cada clase para cada observación. La primera columna es la probabilidad de "No" y la segunda de "Yes".

In [ ]:
# Obtener las probabilidades predichas para cada clase
# Columna 0: P(default = No)
# Columna 1: P(default = Yes)
# Útil para establecer umbrales de decisión personalizados
y_test_prd_proba = logistic_regression.predict_proba(X_test)
y_test_prd_proba

array([[9.97774694e-01, 2.22530580e-03],
       [9.99685906e-01, 3.14094431e-04],
       [9.92513236e-01, 7.48676432e-03],
       ...,
       [9.93064599e-01, 6.93540134e-03],
       [9.97090479e-01, 2.90952143e-03],
       [9.99632217e-01, 3.67783016e-04]], shape=(2500, 2))

## 14. Evaluación del Modelo - Accuracy (Exactitud)

Calculamos la exactitud del modelo: el porcentaje de predicciones correctas sobre el total de predicciones en el conjunto de prueba.

In [ ]:
# Calcular la exactitud (accuracy) del modelo
# Accuracy = (TP + TN) / (TP + TN + FP + FN)
# Indica el porcentaje de predicciones correctas
accuracy_score(y_test, y_test_pred)

0.9744

## 15. Matriz de Confusión

Generamos la matriz de confusión que muestra el desempeño detallado del modelo:

```
                 Predicho No    Predicho Yes
Real No          TN              FP
Real Yes         FN              TP
```

**Interpretación:**
- **Verdaderos Negativos (TN)**: Correctamente predijo "No default"
- **Falsos Positivos (FP)**: Predijo "Default" pero el cliente NO hizo default (Error Tipo I)
- **Falsos Negativos (FN)**: Predijo "No default" pero el cliente SÍ hizo default (Error Tipo II)
- **Verdaderos Positivos (TP)**: Correctamente predijo "Default"

**Contexto de negocio:**
Los Falsos Negativos (FN) son costosos: el banco presta dinero a alguien que hará default.

In [ ]:
# Generar matriz de confusión
# Formato: [[TN, FP],
#           [FN, TP]]
# TN: Verdaderos Negativos, FP: Falsos Positivos
# FN: Falsos Negativos, TP: Verdaderos Positivos
confusion_matrix(y_test, y_test_pred)

array([[2404,   13],
       [  51,   32]])

## 16. Coeficientes del Modelo - Intercepto

El intercepto (β₀) es el término independiente de la ecuación de regresión logística. Representa el log-odds cuando todas las variables predictoras son cero.

In [ ]:
# Obtener el término independiente (β₀) de la ecuación
# Representa el log-odds cuando todas las variables predictoras son 0
# Ecuación: log(p/(1-p)) = β₀ + β₁*balance + β₂*income + β₃*student_Yes
logistic_regression.intercept_

array([-10.96602773])

## 17. Coeficientes del Modelo - Variables Predictoras

Los coeficientes (β₁, β₂, β₃) representan el cambio en el **log-odds** por cada unidad de cambio en cada variable predictora.

**Interpretación:**
- **balance (β₁)**: Efecto del saldo de la tarjeta. Un coeficiente positivo alto indica que mayor saldo → mayor probabilidad de default
- **income (β₂)**: Efecto del ingreso. Generalmente negativo: mayor ingreso → menor probabilidad de default
- **student_Yes (β₃)**: Efecto de ser estudiante. Puede ser positivo o negativo dependiendo del contexto

**Conversión de log-odds a odds ratio:**
- Para interpretar el efecto multiplicativo, calcular: $e^{β_i}$
- Si $e^{β_1} = 1.005$, cada $1 adicional en balance multiplica el odds de default por 1.005

**Nota importante:**
Coeficientes positivos aumentan la probabilidad de default, negativos la disminuyen. La magnitud indica la fuerza de la influencia.

In [ ]:
# Obtener los coeficientes (pesos) de cada variable predictora
# Orden: [β₁_balance, β₂_income, β₃_student_Yes]
# β > 0: aumenta la probabilidad de default
# β < 0: disminuye la probabilidad de default
# |β| grande: mayor influencia en la predicción
logistic_regression.coef_

array([[ 5.71484550e-03,  7.47930251e-06, -6.65954142e-01]])